In [2]:
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from xgboost import XGBClassifier


In [3]:
df=pd.read_csv('/run/media/haroon/Local Disk (D:)/Github/Artificial-Intelligence-Labs/DNA Sequence Classifier/data/synthetic_dna_dataset.csv'  )
df.head()

,Sample_ID,Sequence,GC_Content,AT_Content,Sequence_Length,Num_A,Num_T,Num_C,Num_G,kmer_3_freq,Mutation_Flag,Class_Label,Disease_Risk
0,SAMPLE_1,CTTTCGGGATACTTTTGGGATGGTCTTGGTCAAGGGTTTTAGCCCG...,50.0,50.0,100,22,28,19,31,0.986,0,Bacteria,High
1,SAMPLE_2,TTGACCAAATTTGATTGGAAGTGGTAAGCGCGTATTCCTAGCATCA...,45.0,55.0,100,27,28,22,23,0.486,1,Virus,Medium
2,SAMPLE_3,GCGTGAGTTCTAATTTAAAAAGTCGTAACACGTACCCCGGCGTGTA...,51.0,49.0,100,26,23,30,21,0.367,1,Bacteria,Low
3,SAMPLE_4,ACTACGCGGACAAGAACCAACAGAACCTGGTTTTCGCAAGGGAGTG...,55.0,45.0,100,28,17,23,32,0.404,0,Human,Medium
4,SAMPLE_5,TTCAATGCAGATTGAAAGTTACTTTCATCTGCCCTATGGGTCCCTT...,46.0,54.0,100,24,30,25,21,0.818,0,Human,High


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Sample_ID        3000 non-null   str    
 1   Sequence         3000 non-null   str    
 2   GC_Content       3000 non-null   float64
 3   AT_Content       3000 non-null   float64
 4   Sequence_Length  3000 non-null   int64  
 5   Num_A            3000 non-null   int64  
 6   Num_T            3000 non-null   int64  
 7   Num_C            3000 non-null   int64  
 8   Num_G            3000 non-null   int64  
 9   kmer_3_freq      3000 non-null   float64
 10  Mutation_Flag    3000 non-null   int64  
 11  Class_Label      3000 non-null   str    
 12  Disease_Risk     3000 non-null   str    
dtypes: float64(3), int64(6), str(4)
memory usage: 304.8 KB


In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df.isna().sum()

Sample_ID          0
Sequence           0
GC_Content         0
AT_Content         0
Sequence_Length    0
Num_A              0
Num_T              0
Num_C              0
Num_G              0
kmer_3_freq        0
Mutation_Flag      0
Class_Label        0
Disease_Risk       0
dtype: int64

In [7]:
print(df["Class_Label"].value_counts())
print(df.groupby("Class_Label")[["GC_Content", "Sequence_Length", "kmer_3_freq", "Mutation_Flag"]].mean())
print(pd.crosstab(df["Class_Label"], df["Disease_Risk"]))

Class_Label
Bacteria    761
Human       749
Plant       747
Virus       743
Name: count, dtype: int64
             GC_Content  Sequence_Length  kmer_3_freq  Mutation_Flag
Class_Label                                                         
Bacteria      50.143233            100.0     0.556352       0.505913
Human         50.660881            100.0     0.546921       0.514019
Plant         49.918340            100.0     0.547008       0.464525
Virus         49.753701            100.0     0.540474       0.502019
Disease_Risk  High  Low  Medium
Class_Label                    
Bacteria       267  230     264
Human          252  245     252
Plant          275  240     232
Virus          252  263     228


In [8]:
print(df["Disease_Risk"].value_counts())
print(df.groupby("Disease_Risk")[["GC_Content", "kmer_3_freq", "Mutation_Flag"]].mean())

Disease_Risk
High      1046
Low        978
Medium     976
Name: count, dtype: int64
              GC_Content  kmer_3_freq  Mutation_Flag
Disease_Risk                                        
High           50.360421     0.541249       0.488528
Low            49.770961     0.544901       0.507157
Medium         50.212090     0.557537       0.494877


In [11]:
TARGET='Class_Label'

y=LabelEncoder().fit_transform(df[TARGET])

kmer=CountVectorizer(analyzer='char', ngram_range=(3, 4), lowercase=False)
X_kmer=kmer.fit_transform(df["Sequence"])
X_kmer=X_kmer.multiply(1/X_kmer.sum(axis=1)).tocsr()

num_cols=['GC_Content', 'Sequence_Length', 'kmer_3_freq', 'Mutation_Flag']
X=hstack([X_kmer, csr_matrix(df[num_cols].values)]).tocsr()

cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models={
    "Baseline (Majority)": DummyClassifier(strategy='most_frequent'),
    "LogisticRegression":LogisticRegression(max_iter=1000, random_state=42),
    "XGBoost":XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

for name, m in models.items():
    print(f"{name:20s} {cross_val_score(m, X, y, cv=cv).mean():.3f}")


Baseline (Majority)  0.254
LogisticRegression   0.254


/home/haroon/.global_env/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [13:28:07] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/haroon/.global_env/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [13:28:09] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/haroon/.global_env/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [13:28:10] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/haroon/.global_env/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [13:28:14] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/hom

XGBoost              0.248
